In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis Resume (`bumps-dream`): LBCO, HRPT

This tutorial shows how to reopen the Bayesian project created previously,
inspect the saved fit results and then run more sampling steps to
extend the existing chain. Both BUMPS-DREAM and emcee support saving
and resuming their sampler state, so the same workflow applies to
either engine.

This workflow is useful when:
- the initial sampling run has not yet converged and more steps are needed,
- the initial sampling run has converged but more steps are desired
  for better posterior resolution,
- the initial sampling run has converged but the posterior plots have
  not yet been inspected and the user wants to see the plots before
  deciding whether to run more steps.

The workflow uses the same La0.5Ba0.5CoO3 powder diffraction example
as the DREAM Bayesian tutorial:

- run a short local refinement,
- derive finite fit bounds for the sampled parameters,
- switch to DREAM and sample the posterior,
- save the project with the DREAM sampler state,
- resume the chain with additional steps,
- inspect posterior plots after each sampling stage.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📂 Load Project

### Locate Project

Download and extract the saved DREAM project, with the persisted
sampler state and posterior caches, from the EasyDiffraction data
repository.

In [3]:
project_dir = edi.download_data('proj-lbco-hrpt-dream', destination='projects')

Getting data...


Data 'proj-lbco-hrpt-dream': Bayesian Analysis (bumps-dream): LBCO, HRPT


✅ Data 'proj-lbco-hrpt-dream' downloaded and extracted to '../../../projects/proj-lbco-hrpt-dream-bcff2d3b8ff0'


### Load Project

Loading restores the persisted fit state, posterior samples, and plot
caches. No new fit is launched in this tutorial.

In [4]:
project = edi.Project.load(project_dir)

⚠️ Switching minimizer type removes these settings:                                                                               
   • max_iterations                                                                                                               


⚠️ Switching minimizer type adds these settings with defaults:                                                                    
   • burn_in_steps=600                                                                                                            
   • initialization_method='latin_hypercube'                                                                                      
   • parallel_workers=0                                                                                                           
   • population_size=4                                                                                                            
   • random_seed=None                                                                                                             
   • sampling_steps=3000                                                                                                          
   • thinning_interval=1                                                           

Re-save the project to a fresh working directory so resuming the
chain below writes there instead of the bundled read-only copy.

In [5]:
project.save_as(dir_path='projects/bayesian-dream-resume-lbco-hrpt')

Saving project 📦 'lbco_hrpt_dream' to '../../../projects/bayesian-dream-resume-lbco-hrpt'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 lbco.edi


├── 📁 experiments/


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 mcmc.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_dream.html


## 📊 Inspect Results

### Display Structure

Render the La0.5Ba0.5CoO3 structure restored from the saved project.

In [6]:
project.display.structure(struct_name='lbco')

Structure 🧩 'lbco' (Atom view type: 'covalent')


### Display Fit Results

The fit summary reports the committed point estimate, sampler
settings, convergence diagnostics, and posterior parameter summaries
from the saved Bayesian run.

In [7]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,sampling_steps,10000,Total sampler iterations per chain.
2,burn_in_steps,2000,Sampler iterations discarded as warm-up.
3,thinning_interval,1,Sampler thinning interval.
4,population_size,4,Number of chains or walkers.
5,parallel_workers,0,Worker count; 0 uses all available CPUs.
6,initialization_method,latin_hypercube,Sampler initialization method.
7,random_seed,42,Random seed; None uses a system-derived seed.


📋 Bayesian fit results:


,Metric,Value
1,🧪 Sampler,dream
2,✅ Overall status,success
3,💬 Engine message,DREAM sampling completed
4,⏱️ Fitting time (seconds),1979.34
5,📏 Goodness-of-fit (reduced χ²),1.29
6,"📏 R-factor (Rf, %)",5.65
7,"📏 R-factor squared (Rf², %)",4.91
8,"📏 Weighted R-factor (wR, %)",4.08
9,📉 Best log-posterior,-1157.01
10,📊 Convergence status,passed


📈 Committed parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8913,3.8913,0.0001,0.00 % ↓
2,hrpt,linked_structure,lbco,scale,,9.1329,9.1330,0.0292,0.00 % ↑
3,hrpt,peak,,broad_gauss_u,deg²,0.0817,0.0817,0.0067,0.04 % ↑
4,hrpt,peak,,broad_gauss_v,deg²,-0.1169,-0.1169,0.0048,0.01 % ↓
5,hrpt,instrument,,twotheta_offset,deg,0.6306,0.6306,0.0017,0.00 % ↑


📊 Posterior distribution:


,datablock,category,entry,parameter,units,median,95% CI,r-hat,ess bulk
1,lbco,cell,,length_a,Å,3.8913,"[3.8911, 3.8915]",1.003,7072.4
2,hrpt,linked_structure,lbco,scale,,9.1328,"[9.0754, 9.1903]",1.003,8504.9
3,hrpt,peak,,broad_gauss_u,deg²,0.0814,"[0.0687, 0.0946]",1.003,7256.3
4,hrpt,peak,,broad_gauss_v,deg²,-0.1167,"[-0.1262, -0.1074]",1.003,7393.3
5,hrpt,instrument,,twotheta_offset,deg,0.6303,"[0.6270, 0.6335]",1.003,6949.2


### Display Correlations

The correlation matrix is restored from the saved project state.

In [8]:
project.display.fit.correlations()

### Display Posterior Densities

The pair plot and one-dimensional posterior distributions now load
from the persisted caches generated when the Bayesian fit was saved.

In [9]:
project.display.posterior.pairs()

In [10]:
project.display.posterior.distribution()

### Display Posterior Predictive

The posterior predictive view reuses the cached predictive summary
stored in the project rather than recalculating it on first display.
It overlays the 95% credible interval propagated from the posterior
samples.

In [11]:
project.display.posterior.predictive(expt_name='hrpt')

A zoomed view is useful for checking the propagated uncertainty in a
narrow region of the diffraction pattern.

In [12]:
project.display.posterior.predictive(expt_name='hrpt', x_min=92, x_max=93)

## 🎲 Resume Sampling

### Run Sampling

Resume from the saved DREAM state and append 100 more generations to
the existing chain. We use only 100 steps here to keep the tutorial
fast, but in practice you would typically run more steps to ensure
convergence and better posterior resolution.

Each DREAM generation evaluates the whole population in parallel, so
the cost of resuming scales with `population_size`: `extra_steps=100`
with the default population is on the order of a couple of thousand
model evaluations, not 100. The progress bar counts the new
generations (`1/100`), independent of how long the saved chain
already is.

In [13]:
project.analysis.minimizer.random_seed = 42  # fixed seed for reproducible output
project.analysis.fit(resume=True, extra_steps=100)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'bumps (dream)'...


📈 Bayesian sampling progress:


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 mcmc.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_dream.html


In [14]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,sampling_steps,10000,Total sampler iterations per chain.
2,burn_in_steps,2000,Sampler iterations discarded as warm-up.
3,thinning_interval,1,Sampler thinning interval.
4,population_size,4,Number of chains or walkers.
5,parallel_workers,0,Worker count; 0 uses all available CPUs.
6,initialization_method,latin_hypercube,Sampler initialization method.
7,random_seed,42,Random seed; None uses a system-derived seed.


📋 Bayesian fit results:


,Metric,Value
1,🧪 Sampler,bumps (dream)
2,✅ Overall status,success
3,💬 Engine message,DREAM sampling completed
4,⏱️ Fitting time (seconds),95.04
5,📏 Goodness-of-fit (reduced χ²),1.29
6,"📏 R-factor (Rf, %)",5.65
7,"📏 R-factor squared (Rf², %)",4.92
8,"📏 Weighted R-factor (wR, %)",4.08
9,📉 Best log-posterior,-1157.01
10,📊 Convergence status,passed


📈 Committed parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8913,3.8913,0.0001,0.00 % ↑
2,hrpt,linked_structure,lbco,scale,,9.1330,9.1329,0.0292,0.00 % ↓
3,hrpt,peak,,broad_gauss_u,deg²,0.0817,0.0817,0.0067,0.04 % ↓
4,hrpt,peak,,broad_gauss_v,deg²,-0.1169,-0.1169,0.0048,0.01 % ↑
5,hrpt,instrument,,twotheta_offset,deg,0.6306,0.6306,0.0017,0.00 % ↓


📊 Posterior distribution:


,datablock,category,entry,parameter,units,median,95% CI,r-hat,ess bulk
1,lbco,cell,,length_a,Å,3.8913,"[3.8911, 3.8915]",1.003,7114.3
2,hrpt,linked_structure,lbco,scale,,9.1328,"[9.0754, 9.1903]",1.003,8598.5
3,hrpt,peak,,broad_gauss_u,deg²,0.0814,"[0.0687, 0.0946]",1.003,7353.6
4,hrpt,peak,,broad_gauss_v,deg²,-0.1167,"[-0.1262, -0.1074]",1.003,7491.6
5,hrpt,instrument,,twotheta_offset,deg,0.6303,"[0.6270, 0.6336]",1.003,6995.2


### Display Resumed Posterior

After resume, the posterior plots use the extended chain.

In [15]:
project.display.posterior.pairs()

In [16]:
project.display.posterior.distribution()

In [17]:
project.display.posterior.predictive(expt_name='hrpt', x_min=92, x_max=93)

## 💾 Save Project

In [18]:
project.save_as(dir_path='projects/bayesian-dream-resume-lbco-hrpt')

Saving project 📦 'lbco_hrpt_dream' to '../../../projects/bayesian-dream-resume-lbco-hrpt'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 lbco.edi


├── 📁 experiments/


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 mcmc.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_dream.html
